# Step 4 — Multi-country network with DC power flow

Denmark connected to Germany, Sweden, and Norway via HVAC lines (DC approximation).  
Line capacities are fixed to real ENTSO-E 2015 NTC values; reactance x = 0.1 pu; voltage = 400 kV.

## D1 — Imports and demand data

In [ ]:
# ── CELL D1 ─────────────────────────────────────────────────
# Load hourly demand data for all four countries used in Step D.

from pathlib import Path
import pypsa
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Resolve project root robustly (works whether cwd is project root or subfolder)
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "STEP D - electricity_demand.csv").exists():
    PROJECT_DIR = PROJECT_DIR.parent

demand_path = PROJECT_DIR / "STEP D - electricity_demand.csv"
if not demand_path.exists():
    raise FileNotFoundError(f"Could not find demand file at: {demand_path}")

demand_all = pd.read_csv(
    demand_path,
    sep=";",
    index_col=0,
    parse_dates=True,
)

# Keep only 2015 rows and the four countries we need.
# Convert to timezone-naive (PyPSA requirement).
demand_all.index = pd.to_datetime(demand_all.index, utc=True).tz_localize(None)
demand_all = demand_all.sort_index()
demand_2015 = demand_all[demand_all.index.year == 2015].copy()

# Column names in the CSV use 3-letter country codes.
demand_dk = demand_2015["DNK"].astype(float)
demand_de = demand_2015["DEU"].astype(float)
demand_se = demand_2015["SWE"].astype(float)
demand_no = demand_2015["NOR"].astype(float)

print("Demand shapes (should all be 8760 rows):")
print(f"  DK: {demand_dk.shape},  DE: {demand_de.shape},  SE: {demand_se.shape},  NO: {demand_no.shape}")

assert len(demand_dk) == 8760, "Denmark demand is not 8760 hours."
assert len(demand_de) == 8760, "Germany demand is not 8760 hours."
assert len(demand_se) == 8760, "Sweden demand is not 8760 hours."
assert len(demand_no) == 8760, "Norway demand is not 8760 hours."

print("\nAnnual demand [TWh]:")
print(f"  Denmark: {demand_dk.sum()/1e6:.2f}")
print(f"  Germany: {demand_de.sum()/1e6:.2f}")
print(f"  Sweden:  {demand_se.sum()/1e6:.2f}")
print(f"  Norway:  {demand_no.sum()/1e6:.2f}")

/opt/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Demand shapes (should all be 8760 rows):
  DK: (8760,),  DE: (8760,),  SE: (8760,),  NO: (8760,)

Total annual demand [TWh]:
  Denmark: 32.8
  Germany: 505.3
  Sweden:  135.9
  Norway:  128.7


In [ ]:
# ── CELL D1.5 ───────────────────────────────────────────────
# Load and align hourly capacity factors for Germany, Sweden, and Norway.

if "PROJECT_DIR" not in globals():
    PROJECT_DIR = Path.cwd()
    if not (PROJECT_DIR / "STEP D - electricity_demand.csv").exists():
        PROJECT_DIR = PROJECT_DIR.parent

cf_base = PROJECT_DIR / "Other country data" / "Energy charts data"

# Snapshot reference from Denmark / Step A data.
dataframe_dk = pd.read_csv(PROJECT_DIR / "DK_2015_merged.csv", index_col=0, parse_dates=True)
snapshots = pd.DatetimeIndex(dataframe_dk.index).tz_localize(None)

# Germany: source data is 15-min, convert to hourly by averaging.
cf_de_raw = pd.read_csv(cf_base / "Germany" / "Germany_hourly_capacity_factors.csv")
cf_de_raw["timestamp"] = pd.to_datetime(cf_de_raw["timestamp"])
cf_de_raw = cf_de_raw.set_index("timestamp").sort_index()
cf_de_raw.index = cf_de_raw.index.tz_localize(None)

cf_de = cf_de_raw.resample("h").mean()
cf_de["wind_combined"] = cf_de["Wind onshore"]
cf_de["solar"] = cf_de["Solar AC"]
cf_de["CCGT"] = cf_de["Fossil gas"]
cf_de["nuclear"] = cf_de["Nuclear"]
cf_de = cf_de[["wind_combined", "solar", "CCGT", "nuclear"]].reindex(snapshots)

# Sweden: hourly source data.
cf_se = pd.read_csv(cf_base / "Sweden" / "Sweden_hourly_capacity_factors.csv")
cf_se["timestamp"] = pd.to_datetime(cf_se["timestamp"])
cf_se = cf_se.set_index("timestamp").sort_index()
cf_se.index = cf_se.index.tz_localize(None)

cf_se["wind_combined"] = cf_se["Wind onshore"]
cf_se["nuclear"] = cf_se["Nuclear"]
cf_se["hydro"] = cf_se["Hydro water reservoir"]
cf_se = cf_se[["wind_combined", "nuclear", "hydro"]].reindex(snapshots).bfill()

# Norway: hourly source data.
cf_no = pd.read_csv(cf_base / "Norway" / "Norway_hourly_capacity_factors.csv")
cf_no["timestamp"] = pd.to_datetime(cf_no["timestamp"])
cf_no = cf_no.set_index("timestamp").sort_index()
cf_no.index = cf_no.index.tz_localize(None)

cf_no["wind_combined"] = cf_no["Wind onshore"]
cf_no["hydro"] = cf_no["Hydro water reservoir"]
cf_no = cf_no[["wind_combined", "hydro"]].reindex(snapshots)

print("Germany shape:", cf_de.shape)
print("Sweden shape:", cf_se.shape)
print("Norway shape:", cf_no.shape)

print("\nMissing values:")
print("Germany\n", cf_de.isna().sum())
print("Sweden\n", cf_se.isna().sum())
print("Norway\n", cf_no.isna().sum())

## D2 — Build network and buses

In [ ]:
# ── CELL D2 ─────────────────────────────────────────────────
# Build network: align snapshots and all country-level demand series.

dataframe_dk = pd.read_csv("DK_2015_merged.csv", index_col=0, sep=",", parse_dates=True)
dataframe_dk.index = pd.to_datetime(dataframe_dk.index, utc=True).tz_localize(None)
dataframe_dk = dataframe_dk.sort_index()

CF_wind = dataframe_dk["wind_cf_Unnamed: 1"].astype(float)
CF_solar = dataframe_dk["pv_cf_Unnamed: 1"].astype(float)

# Use Denmark snapshots as the master hourly index.
snapshots = dataframe_dk.index

# Align demand to snapshots.
demand_dk = demand_dk.reindex(snapshots)
demand_de = demand_de.reindex(snapshots)
demand_se = demand_se.reindex(snapshots)
demand_no = demand_no.reindex(snapshots)

assert demand_dk.notna().all(), "Denmark demand has missing values after alignment."
assert demand_de.notna().all(), "Germany demand has missing values after alignment."
assert demand_se.notna().all(), "Sweden demand has missing values after alignment."
assert demand_no.notna().all(), "Norway demand has missing values after alignment."
assert CF_wind.notna().all(), "Wind capacity factor has missing values."
assert CF_solar.notna().all(), "Solar capacity factor has missing values."

net = pypsa.Network()
net.set_snapshots(snapshots)

# Add four buses.
buses = {
    "Denmark": (10.0, 56.0),
    "Germany": (10.5, 51.5),
    "Sweden": (15.0, 59.5),
    "Norway": (10.0, 62.0),
}
for name, (x, y) in buses.items():
    net.add("Bus", name, x=x, y=y)

print("Buses added:", list(net.buses.index))
print(f"Snapshots set: {len(net.snapshots)} hours")

Buses added: ['Denmark', 'Germany', 'Sweden', 'Norway']


## D3 — Add loads

In [3]:
# ── CELL D3 ─────────────────────────────────────────────────
net.add("Load", "load_DK", bus="Denmark", p_set=demand_dk.values)
net.add("Load", "load_DE", bus="Germany", p_set=demand_de.values)
net.add("Load", "load_SE", bus="Sweden",  p_set=demand_se.values)
net.add("Load", "load_NO", bus="Norway",  p_set=demand_no.values)

## D4 — Carriers

In [ ]:
# ── CELL D4 ─────────────────────────────────────────────────
net.add(
    "Carrier",
    ["wind_combined", "solar", "CCGT", "hydro", "nuclear", "coal", "battery"],
    color=["steelblue", "gold", "sienna", "cyan", "purple", "grey", "violet"],
)

print("Carriers added:")
print(list(net.carriers.index))

In [5]:
data = {
    "capital_cost": [
        1500000/25 + 60000,   # wind:  120,000 $/MW/year
        800000/25 + 14000,    # solar:  46,000 $/MW/year
        700000/25 + 24000,    # CCGT:   52,000 $/MW/year
    ],
    "marginal_cost": [0.0, 0.0, 9.5 * 3.6 / 0.56 + 2.30]  # CCGT: ~63.4 $/MWh
}

costs = pd.DataFrame(data, index=["wind_combined", "solar", "CCGT"])
costs.head()

,capital_cost,marginal_cost
wind_combined,120000.0,0.000000
solar,46000.0,0.000000
CCGT,52000.0,63.371429


## D5 — Denmark generators (extendable)

In [6]:
# ── CELL D5 ─────────────────────────────────────────────────
# Denmark generators are extendable; the optimiser sets their capacity.

net.add("Generator", "DK_wind", bus="Denmark", carrier="wind_combined",
        capital_cost     = costs.loc["wind_combined", "capital_cost"],
        marginal_cost    = costs.loc["wind_combined", "marginal_cost"],
        p_max_pu         = CF_wind.values,
        p_nom_extendable = True)

net.add("Generator", "DK_solar", bus="Denmark", carrier="solar",
        capital_cost     = costs.loc["solar", "capital_cost"],
        marginal_cost    = costs.loc["solar", "marginal_cost"],
        p_max_pu         = CF_solar.values,
        p_nom_extendable = True)

net.add("Generator", "DK_CCGT", bus="Denmark", carrier="CCGT",
        capital_cost     = costs.loc["CCGT", "capital_cost"],
        marginal_cost    = costs.loc["CCGT", "marginal_cost"],
        efficiency        = 0.58,
        p_nom_extendable = True)

print("Denmark extendable generators added.")

Denmark extendable generators added.


## D6 — Neighbouring countries (fixed capacity)

Capacities are 2015 installed values from ENTSO-E Statistical Yearbook 2015 and IEA 2015 Electricity Statistics.  
Neighbouring-country generators are **not** extendable — only their dispatch is optimised.

In [ ]:
# ── CELL D6 ─────────────────────────────────────────────────
# Neighbouring countries: fixed-capacity generators, dispatch optimised.

# Germany
net.add("Generator", "DE_wind", bus="Germany", carrier="wind_combined",
        p_nom=41300,
        marginal_cost=0,
        p_max_pu=cf_de["wind_combined"].values,
        p_nom_extendable=False)

net.add("Generator", "DE_solar", bus="Germany", carrier="solar",
        p_nom=37000,
        marginal_cost=0,
        p_max_pu=cf_de["solar"].values,
        p_nom_extendable=False)

net.add("Generator", "DE_CCGT", bus="Germany", carrier="CCGT",
        p_nom=28360,
        marginal_cost=60.0,
        p_nom_extendable=False)

net.add("Generator", "DE_nuclear", bus="Germany", carrier="nuclear",
        p_nom=10800,
        marginal_cost=10.0,
        p_nom_extendable=False)

net.add("Generator", "DE_coal", bus="Germany", carrier="coal",
        p_nom=21420,
        marginal_cost=30.0,
        p_nom_extendable=False)

# Sweden
net.add("Generator", "SE_hydro", bus="Sweden", carrier="hydro",
        p_nom=15920,
        marginal_cost=5.0,
        p_max_pu=cf_se["hydro"].values,
        p_nom_extendable=False)

net.add("Generator", "SE_nuclear", bus="Sweden", carrier="nuclear",
        p_nom=8900,
        marginal_cost=10.0,
        p_nom_extendable=False)

net.add("Generator", "SE_wind", bus="Sweden", carrier="wind_combined",
        p_nom=5500,
        marginal_cost=0,
        p_max_pu=cf_se["wind_combined"].values,
        p_nom_extendable=False)

# Norway
net.add("Generator", "NO_hydro", bus="Norway", carrier="hydro",
        p_nom=29900,
        marginal_cost=5.0,
        p_max_pu=cf_no["hydro"].values,
        p_nom_extendable=False)

net.add("Generator", "NO_wind", bus="Norway", carrier="wind_combined",
        p_nom=700,
        marginal_cost=0,
        p_max_pu=cf_no["wind_combined"].values,
        p_nom_extendable=False)

print("Neighbouring-country generators added (fixed capacities).")
print(net.generators[["bus", "carrier", "p_nom", "p_nom_extendable"]].to_string())

Fixed-capacity generators added.
                bus    p_nom  p_nom_extendable
name                                          
DK_wind     Denmark      0.0              True
DK_solar    Denmark      0.0              True
DK_CCGT     Denmark      0.0              True
DE_wind     Germany  41000.0             False
DE_solar    Germany  39000.0             False
DE_coal     Germany  50000.0             False
DE_CCGT     Germany  28000.0             False
SE_hydro     Sweden  16000.0             False
SE_nuclear   Sweden   9700.0             False
SE_wind      Sweden   6000.0             False
NO_hydro     Norway  32000.0             False
NO_wind      Norway   1100.0             False


## D7 — Transmission lines

Five HVAC lines; two closed loops: **DK–DE–SE–DK** and **DK–NO–SE–DK**.  
Capacities fixed to ENTSO-E 2015 NTC values (MW). Reactance x = 0.1 pu, voltage = 400 kV.

In [ ]:
# ── CELL D7 ─────────────────────────────────────────────────
# Add HVAC transmission lines for the DC approximation.

# Add nominal voltage to buses (required by PyPSA line model conventions).
for bus in net.buses.index:
    net.buses.loc[bus, "v_nom"] = 380

lines = [
    ("line_DK_DE", "Denmark", "Germany", 3500, 360),
    ("line_DK_SE", "Denmark", "Sweden", 1700, 520),
    ("line_DK_NO", "Denmark", "Norway", 1050, 570),
    ("line_SE_NO", "Sweden", "Norway", 3500, 480),
    ("line_DE_SE", "Germany", "Sweden", 600, 820),
]

for name, bus0, bus1, transfer_capacity_mw, length_km in lines:
    net.add(
        "Line",
        name,
        bus0=bus0,
        bus1=bus1,
        s_nom=transfer_capacity_mw,
        x=0.1,
        r=0.01,
        length=length_km,
        p_nom=transfer_capacity_mw,
    )

print("Transmission lines added:")
print(net.lines[["bus0", "bus1", "s_nom", "x", "r", "length"]].to_string())

Transmission lines:
               bus0     bus1   s_nom    x
name                                     
line_DK_DE  Denmark  Germany  1500.0  0.1
line_DK_SE  Denmark   Sweden  1700.0  0.1
line_DK_NO  Denmark   Norway  1050.0  0.1
line_SE_NO   Sweden   Norway  3500.0  0.1
line_DE_SE  Germany   Sweden   600.0  0.1


## D8 — Add batteries and optimise

Step D now uses the updated battery setup from Dimi code (2024 assumptions) and then solves the full 4-country system.

In [ ]:
# ── CELL D8 ─────────────────────────────────────────────────
# Add battery StorageUnits to all countries, then optimize.

print("=== Network Diagnostic ===")
print(f"Buses: {len(net.buses)}, Generators: {len(net.generators)}, Loads: {len(net.loads)}, Lines: {len(net.lines)}, Snapshots: {len(net.snapshots)}")
print()

print("Capacity factors statistics:")
print(f"  CF_wind: min={CF_wind.min():.3f}, max={CF_wind.max():.3f}, mean={CF_wind.mean():.3f}")
print(f"  CF_solar: min={CF_solar.min():.3f}, max={CF_solar.max():.3f}, mean={CF_solar.mean():.3f}")
print()

print("Peak demand by country:")
print(f"  Denmark:   {demand_dk.max()/1e3:.1f} GW (avg: {demand_dk.mean()/1e3:.1f} GW)")
print(f"  Germany:   {demand_de.max()/1e3:.1f} GW (avg: {demand_de.mean()/1e3:.1f} GW)")
print(f"  Sweden:    {demand_se.max()/1e3:.1f} GW (avg: {demand_se.mean()/1e3:.1f} GW)")
print(f"  Norway:    {demand_no.max()/1e3:.1f} GW (avg: {demand_no.mean()/1e3:.1f} GW)")
print()

# Updated Li-ion assumptions (2024) used in Dimi Step D.
battery_investment_power = 100_000
battery_investment_energy = 150_000
battery_fom = 12_500
battery_lifetime = 20
battery_max_hours = 4
battery_efficiency = 0.90
battery_marginal_cost = 0

battery_capital_cost_2024 = (
    battery_investment_power / battery_lifetime
    + battery_fom
    + (battery_investment_energy / battery_lifetime) * battery_max_hours
)
print(f"Battery capital cost (2024): {battery_capital_cost_2024:,.0f} $/MW/year")

for country, bus in [("DK", "Denmark"), ("DE", "Germany"), ("SE", "Sweden"), ("NO", "Norway")]:
    net.add(
        "StorageUnit",
        f"{country}_battery",
        bus=bus,
        carrier="battery",
        capital_cost=battery_capital_cost_2024,
        marginal_cost=battery_marginal_cost,
        efficiency_store=battery_efficiency ** 0.5,
        efficiency_dispatch=battery_efficiency ** 0.5,
        max_hours=battery_max_hours,
        cyclic_state_of_charge=True,
        p_nom_extendable=True,
    )

print("\nAll StorageUnits:")
print(net.storage_units[["bus", "carrier", "max_hours", "capital_cost"]].to_string())
print()

print("Running Step D optimisation...")
try:
    net.consistency_check()
except Exception as exc:
    print(f"Consistency check raised a warning/error: {exc}")

status = net.optimize(solver_name="highs")
print("\nOptimisation finished.")
print("Solver status:", status)

if status[0] == "ok" and net.objective is not None:
    print(f"Objective value: {net.objective/1e9:.3f} B$/year")
    print("\nDenmark optimal capacities [GW]:")
    print((net.generators.loc[["DK_wind", "DK_solar", "DK_CCGT"], "p_nom_opt"] / 1e3).round(3))
    print("\nBattery optimal capacities [MW]:")
    print(net.storage_units[["bus", "p_nom_opt"]].to_string())
else:
    print(f"Optimization failed with status: {status}")

/var/folders/np/y0lp2y655xv_szx354zcxsmh0000gn/T/ipykernel_72794/364436015.py:5: DeprecatedWarning:

df is deprecated as of 1.0.0 and will be removed in 2.0.0. Use `self.components[<component>].static` instead.

/var/folders/np/y0lp2y655xv_szx354zcxsmh0000gn/T/ipykernel_72794/364436015.py:5: DeprecatedWarning:

df is deprecated as of 1.0.0 and will be removed in 2.0.0. Use `self.components[<component>].static` instead.

/var/folders/np/y0lp2y655xv_szx354zcxsmh0000gn/T/ipykernel_72794/364436015.py:5: DeprecatedWarning:

df is deprecated as of 1.0.0 and will be removed in 2.0.0. Use `self.components[<component>].static` instead.

/var/folders/np/y0lp2y655xv_szx354zcxsmh0000gn/T/ipykernel_72794/364436015.py:5: DeprecatedWarning:

df is deprecated as of 1.0.0 and will be removed in 2.0.0. Use `self.components[<component>].static` instead.

/var/folders/np/y0lp2y655xv_szx354zcxsmh0000gn/T/ipykernel_72794/364436015.py:5: DeprecatedWarning:

df is deprecated as of 1.0.0 and will be removed in

Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-ojf03tt8 has 350403 rows; 148923 cols; 564987 nonzeros
Coefficient ranges:
  Matrix  [1e-03, 1e+04]
  Cost    [5e+00, 1e+05]
  Bound   [0e+00, 0e+00]
  RHS     [3e+02, 8e+04]
Presolving model
64494 rows, 103576 cols, 223801 nonzeros  0s
49056 rows, 80776 cols, 169987 nonzeros  0s
Dependent equations search running on 26882 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
48746 rows, 78832 cols, 167113 nonzeros  0s
Presolve reductions: rows 48746(-301657); columns 78832(-70091); nonzeros 167113(-397874) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.3s
      47516     1.4004340099e+10 Pr: 0(0) 1.6s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-p

INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 148923 primals, 350403 duals
Objective: 1.40e+10
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Line-fix-s-lower, Line-fix-s-upper, Kirchhoff-Voltage-Law were not assigned to the network.



Objective value (total system cost): 14.004 B€

Denmark optimal capacities [MW]:
                carrier    p_nom_opt
name                                
DK_wind   wind_combined   918.541126
DK_solar          solar    -0.000000
DK_CCGT            CCGT  1282.998563


## D9 — Results: Denmark capacities vs Step A

In [10]:
# ── CELL D9 ─────────────────────────────────────────────────
print("=== Denmark optimal capacities (Step D) ===")
dk_gens = net.generators[net.generators.bus == "Denmark"]
print(dk_gens[["carrier", "p_nom_opt"]].to_string())

# Compare with Step A (single-node no-interconnection model).
# dnk_n must be in scope from earlier notebook cells.
try:
    print("\n=== Step A (no interconnection) ===")
    print(dnk_n.generators[["carrier", "p_nom_opt"]].to_string())

    key_map = {"DK_wind": "wind_combined", "DK_solar": "solar", "DK_CCGT": "CCGT"}
    print("\n=== Change in Denmark capacities ===")
    for gen, carrier in key_map.items():
        new_val = net.generators.loc[gen, "p_nom_opt"]
        old_val = dnk_n.generators[dnk_n.generators.carrier == carrier]["p_nom_opt"].iloc[0]
        pct = (new_val - old_val) / old_val * 100 if old_val > 0 else float("nan")
        print(f"  {gen}: {old_val/1e3:.2f} GW → {new_val/1e3:.2f} GW  ({pct:+.1f}%)")
except NameError:
    print("(dnk_n not in scope — run Step A cells first to compare)")

=== Denmark optimal capacities (Step D) ===
                carrier    p_nom_opt
name                                
DK_wind   wind_combined   918.541126
DK_solar          solar    -0.000000
DK_CCGT            CCGT  1282.998563

=== Step A (no interconnection) ===
(dnk_n not in scope — run Step A cells first to compare)


## D10 — Line flows

In [11]:
# ── CELL D10 ─────────────────────────────────────────────────
print("=== Average and peak line flow (vs capacity) ===")
avg_flows  = net.lines_t.p0.abs().mean()
peak_flows = net.lines_t.p0.abs().max()
for line in net.lines.index:
    cap  = net.lines.loc[line, "s_nom"]
    print(f"  {line}: avg {avg_flows[line]:.0f} MW  |  peak {peak_flows[line]:.0f} MW  |  limit {cap:.0f} MW  |  avg util {avg_flows[line]/cap*100:.1f}%")

print("\n=== Average electricity price per country [€/MWh] ===")
for bus in net.buses.index:
    avg_price = net.buses_t.marginal_price[bus].mean()
    print(f"  {bus}: {avg_price:.2f} €/MWh")

=== Average and peak line flow (vs capacity) ===
  line_DK_DE: avg 807 MW  |  peak 1500 MW  |  limit 1500 MW  |  avg util 53.8%
  line_DK_SE: avg 1402 MW  |  peak 1700 MW  |  limit 1700 MW  |  avg util 82.5%
  line_DK_NO: avg 1050 MW  |  peak 1050 MW  |  limit 1050 MW  |  avg util 100.0%
  line_SE_NO: avg 392 MW  |  peak 650 MW  |  limit 3500 MW  |  avg util 11.2%
  line_DE_SE: avg 595 MW  |  peak 600 MW  |  limit 600 MW  |  avg util 99.1%

=== Average electricity price per country [€/MWh] ===
  Denmark: 39.96 €/MWh
  Germany: 41.18 €/MWh
  Sweden: 7.77 €/MWh
  Norway: 5.00 €/MWh


---
# Step E — Pen-and-paper verification via PTDF

**Goal:** Using only the network topology and reactances, reproduce the line flows that PyPSA computed for the **first time step** (2015-01-01 00:00).

### Method

Under the DC (linearised AC) approximation the power flow on line $\ell$ is:

$$f_\ell = b_\ell (\theta_{\text{from}(\ell)} - \theta_{\text{to}(\ell)})$$

where $b_\ell = 1/x_\ell$ is the line susceptance and $\theta$ is the voltage angle.

In matrix form for all lines simultaneously:

$$\mathbf{f} = \mathbf{B}_\ell \, \mathbf{K}^\top \boldsymbol{\theta}$$

The nodal power balance gives:

$$\mathbf{p} = \mathbf{B}_{\text{bus}} \boldsymbol{\theta}, \qquad \mathbf{B}_{\text{bus}} = \mathbf{K} \mathbf{B}_\ell \mathbf{K}^\top$$

Eliminating $\boldsymbol{\theta}$ (choosing bus 0 = Denmark as the slack/reference):

$$\mathbf{f} = \underbrace{\mathbf{B}_\ell \, \mathbf{K}_r^\top \, \mathbf{B}_r^{-1}}_{\text{PTDF}} \, \mathbf{p}_r$$

where subscript $r$ means the slack bus row/column is removed.

### Notation

| Symbol | Meaning |
|--------|---------|
| $n$ | number of buses (4: DK, DE, SE, NO) |
| $m$ | number of lines (5) |
| $\mathbf{K}$ | incidence matrix $(n \times m)$: $K_{i\ell}=+1$ if bus $i$ is the *from* end of line $\ell$, $-1$ if *to*, $0$ otherwise |
| $\mathbf{B}_\ell$ | diagonal susceptance matrix $(m \times m)$: $B_{\ell\ell}=1/x_\ell$ |
| $\mathbf{B}_{\text{bus}}$ | nodal susceptance matrix $(n \times n)$ |
| $\mathbf{p}$ | vector of net nodal injections $(n \times 1)$: generation $-$ demand |
| PTDF | Power Transfer Distribution Factor matrix $(m \times (n-1))$ |


## E1 — Incidence matrix **K**

Buses (rows): 0=DK, 1=DE, 2=SE, 3=NO  
Lines (columns): 0=DK→DE, 1=DK→SE, 2=DK→NO, 3=SE→NO, 4=DE→SE

In [12]:
# ── CELL E1 ─────────────────────────────────────────────────
# Build incidence matrix K from the PyPSA network.
# K[i, l] = +1  if bus i is bus0 of line l  (power flows OUT)
# K[i, l] = -1  if bus i is bus1 of line l  (power flows IN)
# K[i, l] =  0  otherwise

bus_order  = list(net.buses.index)          # ['Denmark', 'Germany', 'Sweden', 'Norway']
line_order = list(net.lines.index)          # 5 lines

n = len(bus_order)
m = len(line_order)

K = np.zeros((n, m))
for l_idx, line in enumerate(line_order):
    i_from = bus_order.index(net.lines.loc[line, "bus0"])
    i_to   = bus_order.index(net.lines.loc[line, "bus1"])
    K[i_from, l_idx] = +1
    i_to_idx = bus_order.index(net.lines.loc[line, "bus1"])
    K[i_to_idx,   l_idx] = -1

K_df = pd.DataFrame(K, index=bus_order, columns=line_order)
print("Incidence matrix K  (rows = buses, columns = lines):")
print(K_df.to_string())

Incidence matrix K  (rows = buses, columns = lines):
         line_DK_DE  line_DK_SE  line_DK_NO  line_SE_NO  line_DE_SE
Denmark         1.0         1.0         1.0         0.0         0.0
Germany        -1.0         0.0         0.0         0.0         1.0
Sweden          0.0        -1.0         0.0         1.0        -1.0
Norway          0.0         0.0        -1.0        -1.0         0.0


## E2 — Susceptance matrix **B**ₗ and nodal susceptance matrix **B**_bus

Since all lines have $x = 0.1$ pu, the susceptance of every line is $b = 1/0.1 = 10$ pu.

$$\mathbf{B}_{\text{bus}} = \mathbf{K} \, \mathbf{B}_\ell \, \mathbf{K}^\top$$

In [13]:
# ── CELL E2 ─────────────────────────────────────────────────
# Susceptance of each line: b_l = 1 / x_l
b_values = 1.0 / net.lines["x"].values   # all = 10  (since x = 0.1)
B_l = np.diag(b_values)                  # diagonal matrix (m × m)

# Nodal susceptance matrix
B_bus = K @ B_l @ K.T                    # (n × n)

print("Line susceptances b_l = 1/x_l:", b_values)
print("\nNodal susceptance matrix B_bus (rows/cols = buses):")
print(pd.DataFrame(B_bus, index=bus_order, columns=bus_order).to_string())

Line susceptances b_l = 1/x_l: [10. 10. 10. 10. 10.]

Nodal susceptance matrix B_bus (rows/cols = buses):
         Denmark  Germany  Sweden  Norway
Denmark     30.0    -10.0   -10.0   -10.0
Germany    -10.0     20.0   -10.0     0.0
Sweden     -10.0    -10.0    30.0   -10.0
Norway     -10.0      0.0   -10.0    20.0


## E3 — PTDF matrix

Choose **Denmark (index 0) as the slack bus** (reference, $\theta_0 = 0$).  
Remove its row and column from $\mathbf{B}_{\text{bus}}$ to obtain the reduced system $\mathbf{B}_r$.

$$\text{PTDF}_r = \mathbf{B}_\ell \, \mathbf{K}_r^\top \, \mathbf{B}_r^{-1}$$

This gives a $(m \times (n-1))$ matrix.  
To handle all buses together, prepend a zero column at position 0 (the slack bus injects zero *marginal* PTDF):

$$\text{PTDF} = \begin{bmatrix} \mathbf{0} & \text{PTDF}_r \end{bmatrix} \quad (m \times n)$$

In [14]:
# ── CELL E3 ─────────────────────────────────────────────────
slack_idx = 0  # Denmark = reference bus

# Reduced B_bus: drop row and column of slack bus
B_r = np.delete(np.delete(B_bus, slack_idx, axis=0), slack_idx, axis=1)  # (n-1) × (n-1)

# Reduced incidence matrix: drop the slack-bus column
K_r = np.delete(K, slack_idx, axis=0)   # (n-1) × m  →  we need (m × (n-1))
# Note: K is (n × m), so K.T is (m × n). We drop the slack-bus column from K.T:
KT_r = np.delete(K.T, slack_idx, axis=1)  # (m × (n-1))

# PTDF for non-slack buses
PTDF_r = B_l @ KT_r @ np.linalg.inv(B_r)   # (m × (n-1))

# Full PTDF: insert zero column at slack position
PTDF = np.insert(PTDF_r, slack_idx, 0, axis=1)  # (m × n)

non_slack = [b for b in bus_order if b != bus_order[slack_idx]]
print("PTDF_r  (m × (n-1)):  rows=lines, cols=non-slack buses")
print(pd.DataFrame(PTDF_r, index=line_order, columns=non_slack).round(4).to_string())

print("\nFull PTDF  (m × n): rows=lines, cols=all buses (slack col = 0)")
print(pd.DataFrame(PTDF, index=line_order, columns=bus_order).round(4).to_string())

PTDF_r  (m × (n-1)):  rows=lines, cols=non-slack buses
            Germany  Sweden  Norway
line_DK_DE   -0.625   -0.25  -0.125
line_DK_SE   -0.250   -0.50  -0.250
line_DK_NO   -0.125   -0.25  -0.625
line_SE_NO    0.125    0.25  -0.375
line_DE_SE    0.375   -0.25  -0.125

Full PTDF  (m × n): rows=lines, cols=all buses (slack col = 0)
            Denmark  Germany  Sweden  Norway
line_DK_DE      0.0   -0.625   -0.25  -0.125
line_DK_SE      0.0   -0.250   -0.50  -0.250
line_DK_NO      0.0   -0.125   -0.25  -0.625
line_SE_NO      0.0    0.125    0.25  -0.375
line_DE_SE      0.0    0.375   -0.25  -0.125


## E4 — Nodal injections at first time step

The **net injection** at each bus at time $t=0$ is:

$$p_i(t_0) = \sum_{g \in \text{bus } i} P_g(t_0) - D_i(t_0)$$

where $P_g$ is generator dispatch and $D_i$ is demand.

In [15]:
# ── CELL E4 ─────────────────────────────────────────────────
t0 = net.snapshots[0]
print(f"First time step: {t0}")

# Generation at t0 for each generator
gen_t0 = net.generators_t.p.loc[t0]   # Series, index = generator names

# Demand at t0 for each load
load_t0 = net.loads_t.p_set.loc[t0] if t0 in net.loads_t.p_set.index else net.loads_t.p.loc[t0]

# Aggregate generation per bus
gen_per_bus = pd.Series(0.0, index=bus_order)
for gen in net.generators.index:
    bus = net.generators.loc[gen, "bus"]
    gen_per_bus[bus] += gen_t0[gen]

# Aggregate demand per bus
load_per_bus = pd.Series(0.0, index=bus_order)
for load in net.loads.index:
    bus = net.loads.loc[load, "bus"]
    load_per_bus[bus] += load_t0[load]

# Net injection
p_inj = gen_per_bus - load_per_bus

summary = pd.DataFrame({
    "Generation [MW]": gen_per_bus,
    "Demand [MW]":     load_per_bus,
    "Net injection [MW]": p_inj,
})
print("\nNodal power balance at t0:")
print(summary.round(2).to_string())
print(f"\nSum of injections (must ≈ 0): {p_inj.sum():.4f} MW")

First time step: 2015-01-01 00:00:00

Nodal power balance at t0:
         Generation [MW]  Demand [MW]  Net injection [MW]
Denmark           493.55      3210.98            -2717.43
Germany         44479.71     44546.00              -66.29
Sweden          16662.43     14845.00             1817.43
Norway          16437.29     15471.00              966.29

Sum of injections (must ≈ 0): 0.0000 MW


## E5 — Compute line flows: **f** = PTDF · **p**

With the full PTDF matrix and the nodal injection vector, the line flows are:

$$\mathbf{f} = \text{PTDF} \cdot \mathbf{p}$$

(Positive = flow from bus0 to bus1; negative = reverse direction.)

In [16]:
# ── CELL E5 ─────────────────────────────────────────────────
p_vec = p_inj.values   # (n,) array in bus_order

f_ptdf = PTDF @ p_vec  # (m,) array of line flows

f_ptdf_series = pd.Series(f_ptdf, index=line_order)
print("Line flows from PTDF calculation [MW]:")
for line, flow in f_ptdf_series.items():
    bus0 = net.lines.loc[line, "bus0"]
    bus1 = net.lines.loc[line, "bus1"]
    direction = f"{bus0} → {bus1}" if flow >= 0 else f"{bus1} → {bus0}"
    print(f"  {line}: {flow:+.2f} MW  ({direction})")

Line flows from PTDF calculation [MW]:
  line_DK_DE: -533.71 MW  (Germany → Denmark)
  line_DK_SE: -1133.71 MW  (Sweden → Denmark)
  line_DK_NO: -1050.00 MW  (Norway → Denmark)
  line_SE_NO: +83.71 MW  (Sweden → Norway)
  line_DE_SE: -600.00 MW  (Sweden → Germany)


## E6 — Verification against PyPSA

The PTDF-derived flows should match `net.lines_t.p0` at the first time step **exactly** (within floating-point tolerance), since PyPSA also uses the DC approximation internally.

In [17]:
# ── CELL E6 ─────────────────────────────────────────────────
# PyPSA line flows at t0 (p0 = flow from bus0 to bus1)
f_pypsa = net.lines_t.p0.loc[t0]

comparison = pd.DataFrame({
    "PTDF flow [MW]":  f_ptdf_series.round(4),
    "PyPSA flow [MW]": f_pypsa.round(4),
    "Difference [MW]": (f_ptdf_series - f_pypsa).round(6),
})

print("=== Verification: PTDF vs PyPSA at t0 ===")
print(comparison.to_string())

max_err = (f_ptdf_series - f_pypsa).abs().max()
print(f"\nMax absolute error: {max_err:.6f} MW")
if max_err < 1e-3:
    print("✓ PTDF flows match PyPSA within numerical tolerance.")
else:
    print("⚠ Discrepancy detected — check sign conventions or slack bus assignment.")

=== Verification: PTDF vs PyPSA at t0 ===
            PTDF flow [MW]  PyPSA flow [MW]  Difference [MW]
line_DK_DE       -533.7134        -533.7134             -0.0
line_DK_SE      -1133.7134       -1133.7134             -0.0
line_DK_NO      -1050.0000       -1050.0000              0.0
line_SE_NO         83.7134          83.7134              0.0
line_DE_SE       -600.0000        -600.0000              0.0

Max absolute error: 0.000000 MW
✓ PTDF flows match PyPSA within numerical tolerance.


## E7 — Summary: PTDF interpretation

Each entry $\text{PTDF}_{\ell,i}$ gives the **fraction of 1 MW injected at bus $i$ (and withdrawn at the slack bus)** that flows through line $\ell$.

In [18]:
# ── CELL E7 ─────────────────────────────────────────────────
# Pretty-print the full PTDF with interpretation
print("PTDF matrix (entry = fraction of 1 MW injected at column bus flowing through row line)")
print("Slack bus = Denmark (column shows 0 — all injection referred to DK)\n")

ptdf_df = pd.DataFrame(PTDF, index=line_order, columns=bus_order).round(4)
print(ptdf_df.to_string())

print("\nLine topology reminder:")
print(net.lines[["bus0", "bus1", "s_nom", "x"]].to_string())

PTDF matrix (entry = fraction of 1 MW injected at column bus flowing through row line)
Slack bus = Denmark (column shows 0 — all injection referred to DK)

            Denmark  Germany  Sweden  Norway
line_DK_DE      0.0   -0.625   -0.25  -0.125
line_DK_SE      0.0   -0.250   -0.50  -0.250
line_DK_NO      0.0   -0.125   -0.25  -0.625
line_SE_NO      0.0    0.125    0.25  -0.375
line_DE_SE      0.0    0.375   -0.25  -0.125

Line topology reminder:
               bus0     bus1   s_nom    x
name                                     
line_DK_DE  Denmark  Germany  1500.0  0.1
line_DK_SE  Denmark   Sweden  1700.0  0.1
line_DK_NO  Denmark   Norway  1050.0  0.1
line_SE_NO   Sweden   Norway  3500.0  0.1
line_DE_SE  Germany   Sweden   600.0  0.1
